In [11]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch
from torch.optim import SGD, Adam
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error
from scipy.stats import uniform
from skorch import NeuralNetRegressor
from transform import new_Xtrain, new_Xval, Y_train, Y_val, new_test_df

In [3]:
df = df=pd.read_csv('Project_description_and_data/claims_train.csv')
len(df)

542410

In [4]:
#Preparing the data

num_features = ['Exposure', 'VehPower', 'BonusMalus', 'VehAge_log', 'DrivAge_log', 'Density_log']
scaler = StandardScaler()
scaler.fit(new_Xtrain)
ready_Xtrain = scaler.transform(new_Xtrain)
ready_Xval = scaler.transform(new_Xval)
ready_test_df = scaler.transform(new_test_df)

In [5]:
class ModelDataset(Dataset):
    def __init__(self,x,y):
        self.x = x
        self.y = y
    def __getitem__(self,idx):
        return self.x[idx], self.y[idx]
    
    def __len__(self):
        return len(self.x)

In [6]:
X_train = ready_Xtrain.astype("float32")
X_val = ready_Xval.astype("float32")
Y_train = Y_train.astype("float32")
Y_val = Y_val.astype("float32")

In [7]:
X_train_np = X_train#.to_numpy()
y_train_np = Y_train.to_numpy().reshape(-1, 1)

X_test_np = X_val#.to_numpy()
y_test_np = Y_val.to_numpy().reshape(-1, 1)

ds = ModelDataset(torch.from_numpy(X_train_np),torch.from_numpy(y_train_np))
ds_test = ModelDataset(torch.from_numpy(X_test_np),torch.from_numpy(y_test_np))

train_loader = DataLoader(ds, batch_size=516, shuffle=True)
test_loader = DataLoader(ds_test, batch_size=1, shuffle=True)

In [9]:
#This is our network

input_dim = X_train.shape[1]

class SimpleNN(nn.Module):
    def __init__(self, activation=nn.ReLU):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(input_dim, 36)
        self.fc2 = nn.Linear(36, 24)
        self.fc3 = nn.Linear(24,12)
        self.fc4 = nn.Linear(12, 1)
        self.activation = activation() 
    def forward(self, x):
        #x = x.view(-1, 64)
        x = self.activation(self.fc1(x))
        x = self.activation(self.fc2(x))
        x = self.activation(self.fc3(x))
        x = self.fc4(x)
        return x
    
model = SimpleNN()

In [ ]:
#We perform gridsearch to find the best parameters of the model
param_distributions = {
    'lr': [0.0001, 0.001],
    'optimizer': [Adam],
    'max_epochs': [10, 20, 30, 50],
    'batch_size': [128, 516],
    'module__activation': [nn.ReLU],
    'optimizer__weight_decay': [0.0, 0.0001 ],
}
from skorch import NeuralNetRegressor 
net = NeuralNetRegressor(module=SimpleNN)

grid_search = GridSearchCV(
    net,
    param_distributions,
    cv=3,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    verbose=2
)
grid_search.fit(X_train_np, y_train_np)

print(grid_search.best_params_)
best_net = grid_search.best_estimator_

Fitting 3 folds for each of 32 candidates, totalling 96 fits
  epoch    train_loss    valid_loss      dur
-------  ------------  ------------  -------
      1        0.0599        0.0531  10.5214
      2        0.0550        0.0521  9.9179
      3        0.0539        0.0510  10.6666
      4        0.0528        0.0500  10.0066
      5        0.0517        0.0490  10.4743
      6        0.0508        0.0483  9.4005
      7        0.0501        0.0478  9.3312
      8        0.0497        0.0476  9.7859
      9        0.0495        0.0474  9.9090
     10        0.0493        0.0473  9.4914
     11        0.0492        0.0472  9.6715
     12        0.0491        0.0472  10.1951
     13        0.0490        0.0471  18.8265
     14        0.0489        0.0470  16.2785
     15        0.0489        0.0470  18.7396
     16        0.0488        0.0470  18.5339
     17        0.0487        0.0469  16.4252
     18        0.0487        0.0469  16.7199
     19        0.0486        0.0469  15.5421
 

In [ ]:
#with the best network found we do the prediction on the validation set
y_pred = best_net.predict(ready_Xval.astype("float32"))
mse_val = mean_squared_error(Y_val, y_pred)
r2 = best_net.score(ready_Xval.astype("float32"), Y_val)
print("Validation MSE:", mse_val)
print("Validation R^2:", r2)

Validation MSE: 0.04818841442465782
Validation R^2: 0.1650381088256836


The following is not relevant anymore, it was the model without gridsearch

In [ ]:
#Here we train the network
# criterion = nn.MSELoss()
# optimizer = Adam(model.parameters(), lr=0.0005)

# for epoch in range(50):
#     running_loss = 0.0
#     for i, data in enumerate(train_loader, 0):
#         inputs, labels = data
#         optimizer.zero_grad()
#         outputs = model(inputs.float())
#         loss = criterion(outputs, labels.float())
#         loss.backward()
#         optimizer.step()
#         running_loss += loss.item()
#     print(f"Epoch {epoch + 1}, Loss: {running_loss / len(train_loader)}")

Epoch 1, Loss: 0.055919279294546945
Epoch 2, Loss: 0.05094787175241109
Epoch 3, Loss: 0.04938079760267037
Epoch 4, Loss: 0.04898137868549254
Epoch 5, Loss: 0.04870681842516634
Epoch 6, Loss: 0.04853568305610264
Epoch 7, Loss: 0.048409043582030055
Epoch 8, Loss: 0.0482598140649719
Epoch 9, Loss: 0.048130943421075094
Epoch 10, Loss: 0.04804499825190279
Epoch 11, Loss: 0.0479363669108657
Epoch 12, Loss: 0.047795833785011566
Epoch 13, Loss: 0.04768449340046657
Epoch 14, Loss: 0.047596132504564403
Epoch 15, Loss: 0.04752073532876234
Epoch 16, Loss: 0.04742199040320342
Epoch 17, Loss: 0.04734739425008525
Epoch 18, Loss: 0.047222283942343644
Epoch 19, Loss: 0.04718834373936075
Epoch 20, Loss: 0.04711062035232109
Epoch 21, Loss: 0.04707193789243982
Epoch 22, Loss: 0.046985960504600466
Epoch 23, Loss: 0.046916271797984875
Epoch 24, Loss: 0.04685389724483671
Epoch 25, Loss: 0.046869924502374156
Epoch 26, Loss: 0.0467892187812524
Epoch 27, Loss: 0.04666238720002767
Epoch 28, Loss: 0.0466314890221

In [ ]:
# model.eval()
# with torch.no_grad():
#     outputs = model(torch.from_numpy(X_test_np).float())
#     mse = nn.MSELoss()(outputs, torch.from_numpy(y_test_np).float())
#     rmse = torch.sqrt(mse)
# print("MSE:", mse.item())
# print("RMSE:", rmse.item())

MSE: 0.04774697497487068
RMSE: 0.21851082146167755
